# Pollinex AI — YOLO11 flower detector on a free Colab GPU

Trains the same one-class `flower` detector as `.github/workflows/yolo.yml` (USDA apple/peach/pear + TF Flower Detection sunflower subset, each source keeping its own train/val/test split), but on a GPU and at higher resolution.

**Runtime → Change runtime type → T4 GPU.** Nothing here is pre-run: results appear only after you execute the cells.

Licence: Ultralytics YOLO11 is AGPL-3.0 (see `ai-engine/models/YOLO11_LICENSE_NOTE.md`). Data: CC BY 4.0.

In [ ]:
!nvidia-smi -L || echo 'No GPU - switch the runtime to T4 GPU'
!git clone --depth 1 https://github.com/nvaltryek-001/pollinex-ai.git
%cd pollinex-ai
!pip -q install -r backend/requirements.txt "ultralytics>=8.3" onnx onnxslim

## Train
`--imgsz 1280` keeps small orchard blossoms visible; reduce to 960 if you run out of GPU memory.

In [ ]:
!python ai-engine/training/train_yolo11.py --combined --imgsz 1280 --epochs 100 --batch 8 --device 0

## Evaluate with the project's own metric code (same as the CPU pixel detector)
The test splits were never used for training or model selection.

In [ ]:
ONNX='ai-engine/runs/export/flower_yolo11n.onnx'
!python ai-engine/training/evaluate_detector_on.py --data datasets/processed/usda_fruit_flower --split test --detector pixel --tag pixel
!python ai-engine/training/evaluate_detector_on.py --data datasets/processed/usda_fruit_flower --split test --detector onnx:$ONNX --tag yolo11
!python ai-engine/training/evaluate_detector_on.py --data datasets/processed/tf_flower_detection_sunflower --split test --detector onnx:$ONNX --tag yolo11
!python ai-engine/training/yolo_deploy_rule.py

## Download the results
If the deploy rule passed, `ai-engine/models/flower_yolo11n.onnx` + `.json` exist. Copy them (and `ai-engine/reports/*.json`) into your clone; the backend loads the ONNX model automatically.

In [ ]:
!cd ai-engine && zip -r /content/pollinex_yolo11_results.zip reports models/flower_yolo11n.* runs/export 2>/dev/null; ls -la /content/*.zip
from google.colab import files
files.download('/content/pollinex_yolo11_results.zip')